# 四方向事件流圆检测：逐事件版

这个 Notebook 把 `visualize_circle_detection.py` 的主流程拆开，便于逐段修改和观察：

1. 读取 CSV，并按‘13 点四区域光流图’规则解码、排序；
2. 用一个显式 `for` 循环，每次只向检测器送入一个 `(x, y, c, t)` 事件；
3. 保存指定事件区间的单步输入/输出，并把全部检测结果预计算到数组；
4. 计算完成后，用独立 Cell 启动可拖拽、可调速度、阈值和 xiaoiron 公式参数的事件播放器。

> 可视化固定使用 Qt 桌面后端，会在 Jupyter 外弹出一个独立窗口。请先执行后端 Cell，再执行其余 Cell。

In [1]:
# 必须在导入 matplotlib / 播放器之前选择 Qt 桌面后端。
from IPython import get_ipython

shell = get_ipython()
if shell is None:
    raise RuntimeError("请在 Jupyter/IPython 中运行这个 Notebook")

# %matplotlib qt 会启用 IPython 的 Qt 事件循环，窗口在 Jupyter 页面之外运行。
shell.run_line_magic("matplotlib", "qt")
import matplotlib

backend_name = matplotlib.get_backend()
if "qt" not in backend_name.lower():
    raise RuntimeError(f"未能启用 Qt 后端，当前后端为: {backend_name}")
print(f"Matplotlib backend: {backend_name}（独立桌面窗口）")

Matplotlib backend: qtagg（独立桌面窗口）


In [2]:
from dataclasses import asdict, replace
from pathlib import Path
from pprint import pprint
import sys
import time

import matplotlib.pyplot as plt
import numpy as np

# Notebook 通常从项目目录启动；下面的回退路径保证从其他目录启动时也能导入本项目。
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "four_region_flow.py").exists():
    PROJECT_DIR = Path(r"C:\Users\SynSense\Desktop\小铁项目\new_circle_det")
PROJECT_DIR = PROJECT_DIR.resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import importlib
import circle_detection.xiaoiron_confidence as score_module
import circle_detection.adaptive_detector as detector_module
import visualize_circle_detection as playback_module
# Refresh classes before constructing the new per-event result arrays.
importlib.reload(score_module)
importlib.reload(detector_module)
importlib.reload(playback_module)
from circle_detection.xiaoiron_confidence import XiaoironConfig
from four_region_flow import load_flow_csv
from circle_detection.adaptive_detector import (
    AdaptiveCircleDetector,
    AdaptiveDetectorConfig,
)
from circle_detection.detector import DetectorConfig, DirectionalCircleDetector
from visualize_circle_detection import (
    DIRECTION_ANGLES_DEG,
    DSCTEventPlayer,
    PrecomputedDetections,
    _empty_circle_series,
    _record_detection,
    to_flow_events,
)

print(f"项目目录: {PROJECT_DIR}")

项目目录: C:\Users\SynSense\Desktop\小铁项目\new_circle_det


## 1. 读取 CSV 并预处理

`load_flow_csv` 完成已有的四方向还原：从每行 13 个点的激活状态恢复中心 `(x, y)` 和方向码 `c`，再按时间 `t` 稳定排序。随后转为检测器接收的 `FlowEvent(x, y, c, t)`。

In [3]:
CSV_FILE_NAME = "layer4_20260727_155031_part0001.csv"  # 可改为同目录下的其他 CSV
CSV_PATH = PROJECT_DIR / CSV_FILE_NAME

if not CSV_PATH.exists():
    raise FileNotFoundError(f"找不到数据文件: {CSV_PATH}")

data = load_flow_csv(CSV_PATH)          # 解码 + 时间排序后的 numpy 数据
events_all = to_flow_events(data)       # list[FlowEvent]，保持真实事件顺序

direction_counts = np.bincount(data.direction.astype(int), minlength=4)
print(f"CSV:      {CSV_PATH}")
print(f"事件数:   {len(events_all):,}")
print(f"时长:     {data.duration_s:.6f} s")
print(f"x 范围:   {int(data.x.min())} .. {int(data.x.max())}")
print(f"y 范围:   {int(data.y.min())} .. {int(data.y.max())}")
print(f"方向计数: {dict(enumerate(direction_counts.tolist()))}")
print("方向映射: 0=右下, 1=左下, 2=左上, 3=右上")

# 前 5 个预处理后的事件；event_no 使用从 1 开始的编号。
for event_no, event in enumerate(events_all[:5], start=1):
    print(event_no, event)

CSV:      C:\Users\SynSense\Desktop\小铁项目\new_circle_det\layer4_20260727_155031_part0001.csv
事件数:   155,584
时长:     60.000604 s
x 范围:   1 .. 126
y 范围:   1 .. 126
方向计数: {0: 37041, 1: 38525, 2: 37869, 3: 42149}
方向映射: 0=右下, 1=左下, 2=左上, 3=右上
1 FlowEvent(x=97.0, y=10.0, c=1, t=10299493.0)
2 FlowEvent(x=73.0, y=39.0, c=0, t=10299502.0)
3 FlowEvent(x=11.0, y=17.0, c=0, t=10306817.0)
4 FlowEvent(x=37.0, y=14.0, c=1, t=10307366.0)
5 FlowEvent(x=101.0, y=35.0, c=0, t=10307664.0)


## 2. 设置逐事件计算参数

下列变量都可以直接修改。`END_EVENT=None` 表示处理全部事件；调试算法时可以先设为较小的整数。`INSPECT_*` 只控制保存哪些单步输入/输出，不会影响检测。

In [4]:
# ---------- 运行范围与调试输出 ----------
END_EVENT = None                  # 例如 60000；None = 全部事件
RUN_STRICT_DSCT = True            # False 可只运行当前 adaptive 算法，加快调试
INSPECT_START_EVENT = 53960       # 保存单步记录的起始事件（从 1 开始，含）
INSPECT_END_EVENT = 53980         # 保存单步记录的结束事件（从 1 开始，含）

event_limit = len(events_all) if END_EVENT is None else min(int(END_EVENT), len(events_all))
events = events_all[:event_limit]
if not events:
    raise ValueError("没有可处理的事件")

# ---------- 原始方向约束 DSCT 参数 ----------
strict_config = replace(
    DetectorConfig(),
    max_events=4096,
)

# ---------- 当前 adaptive 三点圆 + 上半圆 + 追踪参数 ----------
# 参数名与 circle_detection/adaptive_detector.py 一致；修改后重跑本 Cell 和计算 Cell。
# xiaoiron 的参数集中放在这里；编号为 0..15，沿用 圆弧.txt。
XIAOIRON_CONFIG = XiaoironConfig(
    occluded_sectors=(10, 11, 12, 13),
    direction_sectors=(1, 2, 5, 6),
    occlusion_gain=0.51,           # 满足证据要求时，缺失扇区奖励最多 +51%
    direction_penalty_lambda=0.85, # exp(-lambda * (1-R))
    min_direction_events=5,       # 1/2/5/6各自达到此原始事件数才算充分
    min_direction_weight=1.75,    # 1/2/5/6各自达到此衰减权重才算充分
    min_other_sectors=6,          # 10/11/12/13以外的有效覆盖要求
    min_sector_weight=1.0,
)
adaptive_config = replace(
    AdaptiveDetectorConfig(),
    window_events=300,             # 最近多少个事件参与几何估计
    decay_events=120.0,            # 事件年龄的指数衰减常数
    min_events=80,                 # 开始检测前的最小事件数
    hypotheses=64,                 # 每次更新随机采样的三点圆数量
    refine_candidates=10,          # 精修候选数量
    refine_iterations=4,
    update_interval_events=6,      # 每 6 个输入事件重算一次；其余事件复用上次结果
    min_radius_px=4.0,
    max_radius_px=90.0,
    radial_tolerance_px=2.5,
    max_radial_mad_px=1.8,
    min_inliers=36,
    min_confidence=0.18,
    allow_upper_arc=True,          # 启用被球杆遮挡时的上半圆验证
    upper_min_inliers=28,
    upper_min_angular_sectors=5,
    upper_min_span_deg=125.0,
    upper_min_confidence=0.14,
    smoothing=0.22,                # 追踪平滑系数
    max_center_jump_px=7.0,
    max_radius_jump_px=4.5,
    jump_confirmations=3,          # 跳变候选连续出现几次才接受
    track_reset_updates=3,
    xiaoiron=XIAOIRON_CONFIG,
)

print(f"本次处理事件: {len(events):,}")
print("adaptive_config：")
pprint(asdict(adaptive_config), sort_dicts=False)

本次处理事件: 155,584
adaptive_config：
{'width': 128,
 'height': 128,
 'window_events': 300,
 'decay_events': 120.0,
 'min_events': 80,
 'hypotheses': 64,
 'refine_candidates': 10,
 'refine_iterations': 4,
 'update_interval_events': 6,
 'min_radius_px': 4.0,
 'max_radius_px': 90.0,
 'centre_margin_px': 8.0,
 'radial_tolerance_px': 2.5,
 'max_radial_mad_px': 1.8,
 'min_inliers': 36,
 'min_inlier_weight_ratio': 0.24,
 'angular_sector_count': 16,
 'min_angular_sector_weight': 1.0,
 'min_angular_sectors': 7,
 'min_quadrants': 3,
 'allow_upper_arc': True,
 'upper_min_inliers': 28,
 'upper_min_inlier_weight_ratio': 0.18,
 'upper_min_angular_sectors': 5,
 'upper_min_span_deg': 125.0,
 'upper_endpoint_radius_ratio': 0.32,
 'upper_apex_radius_ratio': 0.58,
 'upper_confidence_scale': 1.12,
 'min_direction_agreement': 0.05,
 'min_confidence': 0.18,
 'upper_min_confidence': 0.14,
 'continuation_inlier_scale': 0.8,
 'continuation_ratio_scale': 0.85,
 'continuation_sector_relaxation': 1,
 'continuation_co

## 3. 显式 `for` 循环：一次只输入一个事件

循环中的 `step_input` 是本步唯一的新输入；`adaptive_output` / `strict_output` 是本步输出。检测器内部窗口保留旧事件，所以它仍然是有状态、事件驱动的流式计算。当前 adaptive 算法每个事件都会 `push`，每隔 `update_interval_events` 个事件才重算圆，其余步返回上一次结果。

In [5]:
def detection_columns(prefix, detection):
    """把一个检测结果展开，便于逐列查看；None 表示本步没有有效圆。"""
    columns = {
        f"{prefix}_found": detection is not None,
        f"{prefix}_cx": np.nan,
        f"{prefix}_cy": np.nan,
        f"{prefix}_radius": np.nan,
        f"{prefix}_confidence": np.nan,
        f"{prefix}_inliers": 0,
        f"{prefix}_support": "none",
    }
    if detection is not None:
        columns.update({
            f"{prefix}_cx": float(detection.cx),
            f"{prefix}_cy": float(detection.cy),
            f"{prefix}_radius": float(detection.radius),
            f"{prefix}_confidence": float(detection.confidence),
            f"{prefix}_inliers": int(detection.inlier_count),
            f"{prefix}_support": getattr(detection, "support_mode", "unknown"),
        })
    diagnostic = getattr(detection, "xiaoiron", None)
    if diagnostic is not None:
        columns.update({
            "full_confidence": diagnostic.full_confidence,
            "xiaoiron_confidence": diagnostic.xiaoiron_confidence,
            "occlusion_factor": diagnostic.occlusion_factor,
            "direction_factor": diagnostic.direction_factor,
            "evidence_strength": diagnostic.evidence_strength,
            "direction_evidence_R": diagnostic.direction_evidence,
            "direction_gap_1_minus_R": diagnostic.direction_gap,
            "sufficiency_q1_q2_q5_q6": diagnostic.side_sufficiency.tolist(),
            "direction_evidence_r1_r2_r5_r6": diagnostic.side_direction_evidence.tolist(),
            "empty_visible_count": diagnostic.empty_visible_count,
            "counts_10_11_12_13": diagnostic.sector_counts[[10, 11, 12, 13]].tolist(),
            "purity_1_2_5_6": diagnostic.sector_purity[[1, 2, 5, 6]].tolist(),
        })
    return columns


# 每次重跑该 Cell 都创建全新的检测器，保证结果不受上一次运行的内部状态影响。
strict_detector = DirectionalCircleDetector(strict_config, DIRECTION_ANGLES_DEG)
adaptive_detector = AdaptiveCircleDetector(adaptive_config, DIRECTION_ANGLES_DEG)
strict_series = _empty_circle_series(len(events))
adaptive_series = _empty_circle_series(len(events))

last_strict_output = None
debug_rows = []
update_interval = max(1, adaptive_config.update_interval_events)
progress_interval = max(1, len(events) // 20)
started = time.perf_counter()

for event_index, event in enumerate(events):
    event_no = event_index + 1

    # ===== 本步输入：恰好一个事件 (x, y, c, t) =====
    step_input = {
        "event_no": event_no,
        "x": float(event.x),
        "y": float(event.y),
        "c": int(event.c),
        "t_us": float(event.t),
    }

    # ===== 状态更新：所有事件均按到达顺序送入 =====
    if RUN_STRICT_DSCT:
        strict_detector.push(event)
    adaptive_detector.push(event)

    # ===== 本步计算 =====
    # strict DSCT 与 adaptive 使用相同更新节奏，末事件强制补算一次。
    strict_recomputed = RUN_STRICT_DSCT and (
        event_index % update_interval == 0 or event_index == len(events) - 1
    )
    if strict_recomputed:
        last_strict_output = strict_detector.detect(event.t)
    strict_output = last_strict_output

    # detect() 会在未到更新间隔时直接返回缓存结果，不会丢掉刚 push 的事件。
    adaptive_recomputed = event_index % update_interval == 0
    adaptive_output = adaptive_detector.detect()

    # ===== 本步输出持久化：每个事件都有一个对应槽位 =====
    _record_detection(strict_series, event_index, strict_output)
    _record_detection(adaptive_series, event_index, adaptive_output)

    # 仅保存选定区间的逐步 I/O，避免为十几万个事件创建大量 Python 字典。
    if INSPECT_START_EVENT <= event_no <= INSPECT_END_EVENT:
        debug_rows.append({
            **step_input,
            "strict_recomputed": strict_recomputed,
            "adaptive_recomputed": adaptive_recomputed,
            **detection_columns("strict", strict_output),
            **detection_columns("adaptive", adaptive_output),
        })

    if event_no % progress_interval == 0 or event_no == len(events):
        elapsed = time.perf_counter() - started
        print(
            f"\r计算 {event_no:,}/{len(events):,} "
            f"({event_no / len(events):6.1%}), {elapsed:6.1f}s",
            end="\n" if event_no == len(events) else "",
            flush=True,
        )

# 播放器只读这些预计算数组；拖动进度条时不再重新运行检测器。
precomputed = PrecomputedDetections(strict=strict_series, adaptive=adaptive_series)
elapsed = time.perf_counter() - started
adaptive_valid = int(np.isfinite(adaptive_series.confidence).sum())
strict_valid = int(np.isfinite(strict_series.confidence).sum())
print(f"完成：{len(events):,} 个事件，耗时 {elapsed:.2f}s")
print(f"有 adaptive 输出的事件槽: {adaptive_valid:,}")
print(f"有 strict 输出的事件槽:   {strict_valid:,}")

计算 155,584/155,584 (100.0%),   74.1s
完成：155,584 个事件，耗时 74.12s
有 adaptive 输出的事件槽: 122,638
有 strict 输出的事件槽:   384


### 查看循环中保存的单步输入/输出

每一行对应一个输入事件。`*_recomputed=False` 表示本步仍接收了事件，但为节省算力直接复用了上次圆结果。修改 `INSPECT_START_EVENT` / `INSPECT_END_EVENT` 后，重跑参数 Cell 和循环 Cell 即可查看其他区间。

In [6]:
if not debug_rows:
    print("当前检查区间不在已处理范围内，请调整 INSPECT_START_EVENT / INSPECT_END_EVENT。")
else:
    try:
        import pandas as pd
        display(pd.DataFrame(debug_rows))
    except ImportError:
        print("未安装 pandas，改用字典形式显示：")
        for row in debug_rows:
            pprint(row)

,event_no,x,y,c,t_us,strict_recomputed,adaptive_recomputed,strict_found,strict_cx,strict_cy,...,occlusion_factor,direction_factor,evidence_strength,direction_evidence_R,direction_gap_1_minus_R,sufficiency_q1_q2_q5_q6,direction_evidence_r1_r2_r5_r6,empty_visible_count,counts_10_11_12_13,purity_1_2_5_6
0,53960,78.0,76.0,2,31322967.0,False,False,False,NaN,NaN,...,1.0,0.427415,0.0,0.0,1.0,"[0.18656791944885556, 0.0, 1.0, 1.0]","[0.18656791944885556, 0.0, 0.5478435789350405,...",4,"[0, 0, 0, 0]","[1.0, nan, 0.5478435789350405, 0.5528603179455..."
1,53961,97.0,62.0,1,31323283.0,False,False,False,NaN,NaN,...,1.0,0.427415,0.0,0.0,1.0,"[0.18656791944885556, 0.0, 1.0, 1.0]","[0.18656791944885556, 0.0, 0.5478435789350405,...",4,"[0, 0, 0, 0]","[1.0, nan, 0.5478435789350405, 0.5528603179455..."
2,53962,74.0,53.0,1,31323584.0,False,False,False,NaN,NaN,...,1.0,0.427415,0.0,0.0,1.0,"[0.18656791944885556, 0.0, 1.0, 1.0]","[0.18656791944885556, 0.0, 0.5478435789350405,...",4,"[0, 0, 0, 0]","[1.0, nan, 0.5478435789350405, 0.5528603179455..."
3,53963,96.0,61.0,1,31323932.0,False,False,False,NaN,NaN,...,1.0,0.427415,0.0,0.0,1.0,"[0.18656791944885556, 0.0, 1.0, 1.0]","[0.18656791944885556, 0.0, 0.5478435789350405,...",4,"[0, 0, 0, 0]","[1.0, nan, 0.5478435789350405, 0.5528603179455..."
4,53964,94.0,59.0,1,31323934.0,False,False,False,NaN,NaN,...,1.0,0.427415,0.0,0.0,1.0,"[0.18656791944885556, 0.0, 1.0, 1.0]","[0.18656791944885556, 0.0, 0.5478435789350405,...",4,"[0, 0, 0, 0]","[1.0, nan, 0.5478435789350405, 0.5528603179455..."
5,53965,95.0,60.0,1,31323937.0,True,True,False,NaN,NaN,...,1.0,0.427415,0.0,0.0,1.0,"[0.1774688946476304, 0.0, 1.0, 1.0]","[0.1774688946476304, 0.0, 0.6521548871994214, ...",4,"[0, 0, 0, 0]","[1.0, nan, 0.6521548871994214, 0.5815021679357..."
6,53966,78.0,82.0,2,31324310.0,False,False,False,NaN,NaN,...,1.0,0.427415,0.0,0.0,1.0,"[0.1774688946476304, 0.0, 1.0, 1.0]","[0.1774688946476304, 0.0, 0.6521548871994214, ...",4,"[0, 0, 0, 0]","[1.0, nan, 0.6521548871994214, 0.5815021679357..."
7,53967,103.0,79.0,2,31324411.0,False,False,False,NaN,NaN,...,1.0,0.427415,0.0,0.0,1.0,"[0.1774688946476304, 0.0, 1.0, 1.0]","[0.1774688946476304, 0.0, 0.6521548871994214, ...",4,"[0, 0, 0, 0]","[1.0, nan, 0.6521548871994214, 0.5815021679357..."
8,53968,104.0,77.0,1,31325516.0,False,False,False,NaN,NaN,...,1.0,0.427415,0.0,0.0,1.0,"[0.1774688946476304, 0.0, 1.0, 1.0]","[0.1774688946476304, 0.0, 0.6521548871994214, ...",4,"[0, 0, 0, 0]","[1.0, nan, 0.6521548871994214, 0.5815021679357..."
9,53969,103.0,76.0,1,31325518.0,False,False,False,NaN,NaN,...,1.0,0.427415,0.0,0.0,1.0,"[0.1774688946476304, 0.0, 1.0, 1.0]","[0.1774688946476304, 0.0, 0.6521548871994214, ...",4,"[0, 0, 0, 0]","[1.0, nan, 0.6521548871994214, 0.5815021679357..."


## 4. 计算完成后，在独立桌面窗口启动播放器

画面按 CSV 的 `timestamp` 推进：1× 时 1 秒录制时间对应 1 秒现实时间，最低可减慢到 0.01×。事件密集时同一刷新帧会出现多个事件，稀疏时会按真实时间等待；旧事件也按时间戳逐渐变淡。播放器中的 `Loop range` 双端滑块用于选择事件区间，点击 `Loop: Off` 切换为 `Loop: On` 后即可反复播放该区间。右方向键仍可逐事件单步。右侧可以实时修改 xiaoiron 公式参数：它会立即重算缓存证据上的整条置信度曲线，但不会重跑圆几何检测与追踪。

In [7]:
# 自动重新加载播放器实现，避免 Jupyter 继续使用修改前的缓存类。
import importlib
import visualize_circle_detection as playback_module
playback_module = importlib.reload(playback_module)
DSCTEventPlayer = playback_module.DSCTEventPlayer

# ---------- 播放参数（修改后只需重跑本 Cell） ----------
START_EVENT = 1
TRAIL_MS = 160.0                 # 屏幕中保留最近多少毫秒的事件
FADE_TAU_MS = 50.0               # 时间衰减常数；越小，旧事件变淡越快
INTERVAL_MS = 30                 # 画面刷新间隔，只影响流畅度，不改变录制时间
PLAYBACK_SPEED = 1.0             # 时间倍率：1.0=实时，2.0=二倍速，0.5=半速
CONFIDENCE_THRESHOLD = 0.18      # 低于此值的圆不显示；播放器中可拖动
CONFIDENCE_METRIC = "xiaoiron_confidence"  # 或 "confidence" 对照旧评分
SHOW_SECTORS = True              # 画面显示扇区编号；S 键开关
HISTORY_EVENTS = 600             # 圆心/半径历史曲线长度

# 重跑本 Cell 时停止旧动画，避免存在两个计时器。
if "player" in globals():
    old_animation = getattr(player, "animation", None)
    if old_animation is not None and old_animation.event_source is not None:
        old_animation.event_source.stop()
    plt.close(player.figure)

start_index = int(np.clip(START_EVENT - 1, 0, len(events) - 1))
player = DSCTEventPlayer(
    data,
    events,
    precomputed,
    trail_ms=TRAIL_MS,
    fade_tau_ms=FADE_TAU_MS,
    interval_ms=INTERVAL_MS,
    playback_speed=PLAYBACK_SPEED,
    confidence_threshold=CONFIDENCE_THRESHOLD,
    history_events=HISTORY_EVENTS,
    start_index=start_index,
    enable_timer=True,
    confidence_metric=CONFIDENCE_METRIC,
    show_sectors=SHOW_SECTORS,
    xiaoiron_config=XIAOIRON_CONFIG,
    score_window_events=adaptive_config.window_events,       # 播放器右侧 score N 的初始值
    score_decay_events=adaptive_config.decay_events,
    score_radial_tolerance_px=adaptive_config.radial_tolerance_px,
    score_update_interval_events=adaptive_config.update_interval_events,
)

# 保留全局 player 引用，防止动画对象被 Python 回收。
plt.show(block=False)
player.figure.canvas.draw_idle()

# 尽量把新窗口带到前台；某些 Windows 窗口策略可能会让它在任务栏闪烁。
try:
    qt_window = player.figure.canvas.manager.window
    qt_window.show()
    qt_window.raise_()
    qt_window.activateWindow()
except AttributeError:
    pass

print("播放器已在独立 Qt 窗口启动。")

播放器已在独立 Qt 窗口启动。


## 5. 检查任意事件的评分依据
以下统计来自检测器最近 300 个事件的窗口，采用事件序号衰减；它与播放器的毫秒残影窗口不同。新分数用 full 表达式作为基础，即使该几何由 upper 模式接受，也不叠加 upper 奖励。原有几何筛选和追踪先产生候选，新分数用于候选输出和显示阈值，不代表已确认击球。
当前采用 R=min(q1*p1,q2*p2,q5*p5,q6*p6)，空缺集合为10/11/12/13。圆几何检测与事件证据首次仍需预计算；播放器右侧调整 alpha、lambda、N_min、W_min、K_min 和扇区权重阈值时，会直接用缓存证据实时重算评分，无需重新执行事件循环。

In [8]:
INSPECT_EVENT = 141697  # 从 1 开始；可改为任意需要检查的事件
i = int(np.clip(INSPECT_EVENT - 1, 0, len(events) - 1))
series = precomputed.adaptive
print('event:', i + 1, 'full:', series.full_confidence[i], 'xiaoiron:', series.xiaoiron_confidence[i])
print('occlusion factor:', series.occlusion_factor[i], 'direction factor:', series.direction_factor[i], 'G:', series.evidence_strength[i])
print('q1,q2,q5,q6:', series.side_sufficiency[i].tolist(), 'r1,r2,r5,r6:', series.side_direction_evidence[i].tolist(), 'R:', series.direction_evidence[i])
hist = series.direction_weights[i]
rows = []
for s in range(16):
    mass = float(hist[s].sum())
    rows.append({'sector': s, 'raw_count': int(series.sector_counts[i, s]),
                 'weights_c0_c1_c2_c3': hist[s].tolist(),
                 'purity': float(hist[s].max()/mass) if mass else None,
                 'arc_in_sensor': bool(series.sector_visible[i, s])})
pprint(rows, sort_dicts=False)


event: 141697 full: 0.34593853 xiaoiron: 0.40796247
occlusion factor: 1.4348321 direction factor: 0.8219023 G: 0.85261196
q1,q2,q5,q6: [1.0, 1.0, 0.8526119589805603, 1.0] r1,r2,r5,r6: [0.8855584859848022, 0.9319740533828735, 0.7692543864250183, 0.930695116519928] R: 0.7692544
[{'sector': 0,
  'raw_count': 7,
  'weights_c0_c1_c2_c3': [0.3899773359298706, 0.0, 2.5153584480285645, 0.0],
  'purity': 0.865772008895874,
  'arc_in_sensor': True},
 {'sector': 1,
  'raw_count': 22,
  'weights_c0_c1_c2_c3': [0.0,
                          0.7620745897293091,
                          8.889634132385254,
                          0.3867410123348236],
  'purity': 0.8855584859848022,
  'arc_in_sensor': True},
 {'sector': 2,
  'raw_count': 18,
  'weights_c0_c1_c2_c3': [0.0, 0.0, 9.260416030883789, 0.6759293675422668],
  'purity': 0.9319740533828735,
  'arc_in_sensor': True},
 {'sector': 3,
  'raw_count': 20,
  'weights_c0_c1_c2_c3': [0.0,
                          0.9355069994926453,
                